In [ ]:
import numpy as np
import pandas as pd
import os
import matplotlib.pyplot as plt

In [ ]:
domain = "Base_Fine_Tuned"
indices = [i for i in range(12)]
type = "Multi_Class_Augmentation"
matrix_name = "Task_Matrix_W.json"
folder_path = f"../Data/{type}/"
num_datasets = [i for i in range(2,9)]

In [ ]:
task_matrixes = []

for i in num_datasets:
    data_path = f"../Data/Multi_Class_Augmentation/{i}_{domain}/Entire_Transformation_Matrix_W"
    try:
        for filename in os.listdir(data_path):
            if filename == ".DS_Store":
                continue
            if filename in ["Task_Matrix_W.json"]:
                file_path = os.path.join(f"{data_path}", filename)
                if os.path.isfile(file_path):
                    task_matrixes.append(file_path)
    except FileNotFoundError:
        print(f"Error: The Folder '{folder_path}' was not found.")
    except Exception as e:
        print(f"An error occured: {e}")

In [ ]:
task_matrixes = [pd.read_json(i) for i in task_matrixes]

In [ ]:
U = {}
S = {}
Vh = {}

for i in indices:
    U[i] = []
    S[i] = []
    Vh[i] = []
    for j in task_matrixes:
        arr = np.array(j["W"][i])
        u, s, vh = np.linalg.svd(arr)
        U[i].append(np.array(u))
        S[i].append(np.array(s))
        Vh[i].append(np.array(vh))
        

In [ ]:
plt.figure(figsize=(8,4))
plt.plot(S[11][6], marker='.', linewidth=1)
plt.title('Singular values (linear scale)')
plt.xlabel('index i')
plt.ylabel(r'$\sigma_i$')
plt.grid(True)

plt.figure(figsize=(8,4))
plt.semilogy(S[11][6], marker='.', linewidth=1)
plt.title('Singular values (log scale)')
plt.xlabel('index i')
plt.ylabel(r'$\sigma_i$ (log scale)')
plt.grid(True)
plt.show()

In [ ]:
energy = S[11][6]**2
cumulative = np.cumsum(energy)
total = cumulative[-1]
frac = cumulative / total  # fraction of Frobenius energy captured

# Pick k for a threshold
threshold = 0.95
k = np.searchsorted(frac, threshold) + 1  # +1 because searchsorted returns idx
print("k for {:.0%} energy:".format(threshold), k)

# Plot fraction
plt.figure(figsize=(8,4))
plt.plot(frac, linewidth=2)
plt.axhline(threshold, color='red', linestyle='--')
plt.xlabel('k')
plt.ylabel('Fraction of energy captured')
plt.grid(True)
plt.show()


In [ ]:
# Operator Norm
print(np.linalg.norm(S[11][6], ord=2))